> **Meridian AgentOps workshop · notebook 05 of 6 (Modules 9–11).** The reusable code lives in the
> repo's `app/` package (agent, tools, MCP service desk, evaluators, config) — these notebooks
> import it, so a fresh session only needs the bootstrap cells below instead of re-running
> earlier modules. **Prerequisite:** notebooks 03 + 04 have run today (v2 shipped + production traffic flowing).


## Module 9 · Monitoring & alerting

a scheduled-job-style health check that compares live production metrics against a saved
baseline and fires an alert on drift — then a real incident: a "harmless" prompt tweak ships, quality tanks,
the alert catches it

The design: **baseline** (what healthy looks like, saved to a file) → **check** (pull the last window's
numbers, compute relative drift, threshold) → **alert** (print/webhook — in real life: Slack/PagerDuty).

In [ ]:
# ── 0.1 Get the code + the pinned stack (fresh Colab/Kaggle VM: clone first) ──
import os
if not os.path.isdir("../app"):                    # fresh cloud VM → clone the repo
    !git clone https://github.com/kartik-nighania/data-hack-summit-2026.git _workshop_repo
    %cd _workshop_repo/workshop
%pip install -q -r ../requirements.txt
print("✅ stack ready — if pip just upgraded packages, do Run ▸ Restart session once and rerun from the top.")


In [ ]:
import os, sys, json, time
from datetime import datetime, timedelta, timezone

sys.path.insert(0, os.path.abspath(".."))        # make the repo's app/ package importable

# Jupyter kernels already run an event loop; this lets libraries that call
# asyncio.run()/run_until_complete work inside notebook cells.
import nest_asyncio
nest_asyncio.apply()

print("✅ environment prepared |", sys.version.split()[0])


In [ ]:
# ── 0.3 Load API keys (Kaggle Secrets → Colab Secrets → .env / env vars → prompt) ─
from app.config import load_keys
load_keys()


In [ ]:
# ── 0.4 Constants (config.yaml) + the Langfuse client (PII masking hook registered) ─
from app.config import AGENT_MODEL, DATASET_NAME, EXPERIMENT_CONCURRENCY, INGESTION_WAIT_S, TRAFFIC_SESSIONS, get_lf
lf = get_lf()
if not lf.auth_check():
    raise SystemExit("❌ Langfuse authentication FAILED. Check keys + LANGFUSE_HOST region "
                     "(EU: https://cloud.langfuse.com / US: https://us.cloud.langfuse.com).")
LANGFUSE_HOST = os.environ["LANGFUSE_HOST"]
import importlib.metadata as _md
print("✅ Langfuse authenticated:", LANGFUSE_HOST)
for p in ["langfuse", "langchain", "langgraph", "deepeval", "openai", "fastmcp"]:
    print(f"   {p}=={_md.version(p)}")


In [ ]:
# Session imports — no local state: the 'v2' prompt LABEL (set by notebook 03) anchors the rollback
import requests
from app.agent import ainvoke_agent
from app.evaluators import RULE_EVALUATORS, make_run_evaluator
from app.prompts import ensure_prompt
from app.generate_fake_traffic import generate_traffic
from app.get_dashboard_metrics import metrics_query
from app.online_eval import score_recent_production

try:
    V2_VERSIONS = {"final-response": lf.get_prompt("final-response", type="chat", label="v2",
                                                   cache_ttl_seconds=0).version}
except Exception:
    raise SystemExit("Run notebook 03 first — it ships v2 and labels it 'v2' for this rollback")

async def golden_task(*, item, **kwargs):
    """One dataset item -> one agent execution (the same task the CI gate runs)."""
    return await ainvoke_agent(item.input["question"], item.input["customer_id"])


In [ ]:
# ── 9.1 Save the healthy baseline ────────────────────────────────────────────
def pull_prod_metrics(minutes_back=120, since=None):
    """3 aggregate API calls - deliberately frugal (Metrics API: 100 req/day on the free tier)."""
    now = datetime.now(timezone.utc); frm = since or (now - timedelta(minutes=minutes_back))
    fts, tts = frm.strftime("%Y-%m-%dT%H:%M:%SZ"), now.strftime("%Y-%m-%dT%H:%M:%SZ")
    obs = metrics_query({"view": "observations",
        "metrics": [{"measure": "totalCost", "aggregation": "sum"},
                    {"measure": "latency", "aggregation": "p95"},
                    {"measure": "count", "aggregation": "count"}],
        "dimensions": [], "timeDimension": None,
        "filters": [{"column": "environment", "operator": "=", "value": "production", "type": "string"}],
        "fromTimestamp": fts, "toTimestamp": tts})
    obs = obs[0] if obs else {}
    def score_avg(name):
        rows = metrics_query({"view": "scores-numeric",
            "metrics": [{"measure": "value", "aggregation": "avg"}, {"measure": "count", "aggregation": "count"}],
            "dimensions": [], "timeDimension": None,
            "filters": [{"column": "name", "operator": "=", "value": name, "type": "string"},
                        {"column": "environment", "operator": "=", "value": "production", "type": "string"}],
            "fromTimestamp": fts, "toTimestamp": tts})
        if not rows:
            return None, 0
        return (float(rows[0]["avg_value"]) if rows[0].get("avg_value") is not None else None,
                int(float(rows[0].get("count_count") or 0)))
    fb, fb_n = score_avg("user-feedback")
    res, res_n = score_avg("resolution_conf")
    return {"window_min": minutes_back,
            "p95_latency_ms": float(obs.get("p95_latency") or 0),
            "total_cost": float(obs.get("sum_totalCost") or 0),
            "n_observations": int(float(obs.get("count_count") or 0)),
            "avg_user_feedback": fb, "n_feedback": fb_n,
            "avg_resolution_conf": res, "n_resolution": res_n}

BASELINE = pull_prod_metrics(minutes_back=120)
with open("baseline.json", "w") as f:
    json.dump({"saved_at": datetime.now(timezone.utc).isoformat(), "metrics": BASELINE}, f, indent=2)
print("✅ healthy baseline saved to baseline.json:\n", json.dumps(BASELINE, indent=2))

In [ ]:
# ── 9.2 The health check (this is your scheduled job) ────────────────────────
WEBHOOK_URL = os.environ.get("MERIDIAN_WEBHOOK_URL", "")     # e.g. a webhook.site URL - optional

THRESHOLDS = {   # relative drift vs baseline that trips an alert
    "avg_user_feedback": -0.25,      # 25% relative drop in 👍-rate
    "avg_resolution_conf": -0.20,    # 20% drop in the online judge's resolution score
    "p95_latency_ms": +0.60,         # 60% latency increase
}

def check_health(minutes_back=30, baseline=None, since=None):
    """Compare a live window against the saved baseline. In production this runs on a schedule
    with a rolling window; after a deploy you compare the WINDOW SINCE THE DEPLOY."""
    baseline = baseline or json.load(open("baseline.json"))["metrics"]
    live = pull_prod_metrics(minutes_back=minutes_back, since=since)
    report, alerts = [], []
    for key, limit in THRESHOLDS.items():
        base, cur = baseline.get(key), live.get(key)
        if base in (None, 0) or cur is None:
            report.append((key, round(base, 3) if isinstance(base, float) else base,
                           round(cur, 3) if isinstance(cur, float) else cur, "NO_DATA", "-")); continue
        drift = (cur - base) / abs(base)
        breached = drift <= limit if limit < 0 else drift >= limit
        status = "🔴 ALERT" if breached else "✅ OK"
        if breached: alerts.append((key, base, cur, drift))
        report.append((key, round(base, 3), round(cur, 3), f"{drift:+.0%}", status))
    print(f"{'metric':<22}{'baseline':>10}{'live':>10}{'drift':>9}   status")
    for row in report:
        print(f"{row[0]:<22}{str(row[1]):>10}{str(row[2]):>10}{str(row[3]):>9}   {row[4]}")
    if alerts and WEBHOOK_URL:
        requests.post(WEBHOOK_URL, json={"type": "meridian-drift-alert",
                                         "alerts": [{"metric": k, "baseline": b, "live": c, "drift": f"{d:+.0%}"}
                                                    for k, b, c, d in alerts],
                                         "at": datetime.now(timezone.utc).isoformat()}, timeout=10)
        print("→ webhook alert POSTed")
    elif alerts:
        print("→ (set MERIDIAN_WEBHOOK_URL - e.g. from webhook.site - to receive this as a webhook)")
    return {"alerts": alerts, "live": live}

health = check_health(minutes_back=120)
print("\nAll green - the baseline window and the live window are the same healthy v2 traffic.")

## The incident: someone ships a 'harmless' brevity tweak (v3)

In [ ]:
# ── 9.3 The incident: someone ships a 'harmless' brevity tweak (v3) ──────────
FINAL_V3 = [
    {"role": "system", "content": (
        "You write the final reply to the customer for Meridian Housing Finance.\n"
        "CRITICAL STYLE RULE: reply in ONE short friendly sentence (maximum ~20 words). Do not include "
        "long numbers, ids, or lists - keep it brief, warm and reassuring. Customers love short answers."
    )},
    {"type": "placeholder", "name": "conversation"},
    {"role": "user", "content": "Now write the single final reply to the customer."},
]
fin3 = ensure_prompt("final-response", FINAL_V3, "chat",
                     labels=["production"], config={"model": AGENT_MODEL, "temperature": 0.3})
lf.update_prompt(name="final-response", version=fin3.version, new_labels=["production"])  # force-promote (idempotent)
print(f"🚀 'final-response' v{fin3.version} is now PRODUCTION (the label moved - no redeploy, remember).")
print("   It looked reasonable in review: shorter answers, happier customers … right?\n")

V3_DEPLOYED_AT = datetime.now(timezone.utc)          # ← monitoring 101: mark the deploy moment
burst_v3 = await generate_traffic(max(8, TRAFFIC_SESSIONS // 3), deploy_version="v3") # smartly generated feedback
print(f"\n⏳ {INGESTION_WAIT_S}s for ingestion, then re-scoring the post-deploy window …"); time.sleep(INGESTION_WAIT_S)
_ = await score_recent_production(since=V3_DEPLOYED_AT, sample=10)
print(f"⏳ {INGESTION_WAIT_S}s for the new scores to become queryable …"); time.sleep(INGESTION_WAIT_S)

In [ ]:
# ── 9.4 The job catches it ───────────────────────────────────────────────────
health = check_health(since=V3_DEPLOYED_AT)      # live window = everything since the v3 deploy
print("""
Read the table: feedback and resolution CRASHED - while p95 latency (and cost) went DOWN.
A cost-only dashboard would have called this deploy an improvement. Quality signals are the alarm.

Diagnose in the UI (2 min): Prompts ▸ final-response ▸ Metrics - v3's generations have collapsed output
tokens;
""")

## 9.5 The native path: Langfuse Monitors (UI, 2 minutes)

Our notebook job is one way; Langfuse also runs this server-side: **Monitors ▸ + New monitor** → data source
**Scores (numeric)** → metric `avg` of `user-feedback` → filter environment = production → lookback **1 hour**
→ alert threshold `< 0.5` → notification via **Automations**: Slack or **Webhook** (paste a
[webhook.site](https://webhook.site) URL to see the signed JSON payload) or GitHub Actions. Free tier: **2
monitors per org** — typically one on feedback, one on cost. Monitors fire on state *transitions* (ALERT,
recovery to OK), so you get one page per incident, not a page per minute.

In [ ]:
# ── 9.6 Rollback = move the label back. Verify recovery. Close the incident. ─
lf.update_prompt(name="final-response", version=V2_VERSIONS["final-response"], new_labels=["production"])
now_prod = lf.get_prompt("final-response", type="chat", cache_ttl_seconds=0)
print(f"⏪ rollback done: production label → v{now_prod.version} (the good v2). One API call, zero deploys.\n")

ROLLED_BACK_AT = datetime.now(timezone.utc)
burst_fix = await generate_traffic(max(6, TRAFFIC_SESSIONS // 4), deploy_version="v2-rollback")
print(f"\n⏳ {INGESTION_WAIT_S}s …"); time.sleep(INGESTION_WAIT_S)
_ = await score_recent_production(since=ROLLED_BACK_AT, sample=8)
print(f"⏳ {INGESTION_WAIT_S}s for the new scores to become queryable …"); time.sleep(INGESTION_WAIT_S)
health = check_health(since=ROLLED_BACK_AT)
print("""
Recovery confirmed by the same job that caught the incident. The full incident timeline - deploy, degradation,
alert, diagnosis, rollback, recovery - is queryable forever: filter by version tag deploy:v3 vs deploy:v2.
In production you'd run check_health() on a schedule (cron / GitHub Actions / Cloud Function) - and that is
exactly the bridge to Module 10.
""")

## Module 10 · Quality gates in CI/CD 

understand how the Module-6 experiment becomes a merge gate: every PR that touches code or
prompts re-runs the evaluation on a **pinned dataset version**, and the merge is blocked if quality drops —
results posted right on the pull request.*

The official recipe is the **`langfuse/experiment-action`** GitHub Action: your repo contains an experiment
script with an `experiment(context)` entrypoint; the action runs it on every PR, posts a score-table comment,
and fails the check when the script raises `RegressionError`.

```yaml
# .github/workflows/eval-gate.yml   (the instructor will run this live)
name: Langfuse experiment gate
on: [pull_request]
permissions: {contents: read, pull-requests: write}
jobs:
  experiment:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v6
      - uses: actions/setup-python@v6
        with: {python-version: "3.12"}
      - uses: langfuse/experiment-action@v1
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
        with:
          langfuse_public_key: ${{ secrets.LANGFUSE_PUBLIC_KEY }}
          langfuse_secret_key: ${{ secrets.LANGFUSE_SECRET_KEY }}
          langfuse_base_url: ${{ vars.LANGFUSE_HOST }}
          experiment_path: tests/run_evals.py
          dataset_name: meridian-golden-v1
          github_token: ${{ github.token }}
```

Three repeatability rules make the gate trustworthy: **freeze the gate dataset** (versioned by NAME —
promoted failures land in the candidates dataset, never here), **pin the judge** (model + versioned judge prompt), and **gate on
aggregates with margin** (a single flaky item must not block a release).

In [ ]:
# ── 10.1 The gate itself, run live (same machinery as the GitHub Action) ─────
from langfuse import RegressionError

GATE_THRESHOLDS = {"avg_completeness": 0.70, "avg_business_rules_ok": 0.90, "avg_route_correct": 0.80}

def run_quality_gate(run_name: str, thresholds=GATE_THRESHOLDS):
    """Re-run the (rule-based) evaluation on the frozen v1 dataset and enforce thresholds.
    In CI this exact logic lives in tests/run_evals.py::experiment(context)."""
    pinned = lf.get_dataset(DATASET_NAME)     # ← the frozen 22-item v1 set (append-never)
    result = pinned.run_experiment(
        name="ci-quality-gate", run_name=run_name,
        description="merge gate on the frozen meridian-golden-v1 set",
        task=golden_task,
        evaluators=RULE_EVALUATORS,                                  # deterministic → cheap, stable gate
        run_evaluators=[make_run_evaluator(s.replace("avg_", "")) for s in thresholds],
        max_concurrency=EXPERIMENT_CONCURRENCY,
    )
    lf.flush()
    aggregates = {e.name: e.value for e in result.run_evaluations if e.value is not None}
    print("gate aggregates:", {k: round(v, 3) for k, v in aggregates.items()})
    for metric, threshold in thresholds.items():
        value = aggregates.get(metric)
        if value is None or value < threshold:
            raise RegressionError(result=result, metric=metric,
                                  value=float(value or 0), threshold=threshold)
    print(f"✅ GATE PASSED - '{run_name}' may merge.")
    return result

gate_result = run_quality_gate("ci-gate-v2")

In [ ]:
# ── 10.2 What a BLOCKED merge looks like (tighten one threshold to force it) ─
try:
    run_quality_gate("ci-gate-strict", thresholds={**GATE_THRESHOLDS, "avg_completeness": 0.99})
except RegressionError as e:
    print(f"⛔ MERGE BLOCKED - RegressionError: {e.metric} = {e.value:.3f} < threshold {e.threshold}")
    print("   In GitHub, this exception fails the check and the action posts the score table on the PR.")

## Module 11 · Wrap-up

The full loop, end to end: **build → trace → version prompts → collect feedback → evaluate
(rules · judges · DeepEval · managed) → evaluate live → dashboards → monitor & alert → CI gate.**

Where to go next: self-hosting Langfuse, wiring the Module-9 job into a real scheduler, and this
repo's own PR gate (`.github/workflows/eval-gate.yml` + `tests/run_evals.py`) as your template —
open a pull request that flips `SABOTAGE_BREVITY` in `app/agent.py` and watch the merge get blocked.
